In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip install --quiet pennylane
!pip install --quiet amazon-braket-pennylane-plugin

# Building a Quantum Circuit
To solve a problem like MNIST (digit categorization) using Quantum Machine Learning, we must move away from "random" unitaries—which are prone to optimization failures like Barren Plateaus—and toward Principled Ansatz Design.In this context, a principled approach means tailoring the circuit architecture to the symmetry and structure of the data. For MNIST, we are dealing with spatial 2D data, which suggests we should derive our Ansatz using principles from Quantum Convolutional Neural Networks (QCNN) or Symmetry-Protected Circuits.

## Step 1: Data Dimensionality Reduction (The "Bottleneck")
A standard MNIST image is $28 \times 28$ pixels (784 features). Using one qubit per pixel is currently impossible. We must first perform a principled dimensionality reduction:
* PCA (Principal Component Analysis): Reducing the 784 features to $N$ components (e.g., $N=4, 8,$ or $16$).
* Autoencoders: Using a classical neural network to compress the image into a "latent space" vector that matches our qubit count.

## Step 2: Deriving the Feature Map (State Preparation)
We don't just "input" numbers; we encode them as rotations. A principled technique for MNIST is the ZZ-Feature Map.
* Why? MNIST digits are defined by the relationship between adjacent pixels (edges and curves).
* The Math: We apply $R_y(x_i)$ for individual pixels, followed by $R_{zz}(x_i, x_j)$ for pairs of pixels. This projects the image into a Hilbert space where spatial correlations become quantum entanglement.

## Step 3: Deriving the Ansatz (Principled Design)
Instead of a random unitary, we derive the Ansatz based on Hardware-Efficiency and Symmetry.
### A. The Multi-Layer Perceptron (MLP) Analogy
We build the Ansatz using "Strongly Entangling Layers." Each layer consists of:Euler Rotations: $R_z(\theta_1) \cdot R_y(\theta_2) \cdot R_z(\theta_3)$ on each qubit. This ensures we can reach any point on the individual Bloch sphere.Entanglers: CNOT gates in a circular or "all-to-all" pattern.
### B. The QCNN Approach (Hierarchical)
If we want to mimic how classical CNNs work for MNIST, we derive a Hierarchical Ansatz:
* Convolutional Layers: Apply two-qubit unitaries $U(\theta)$ to adjacent pairs to extract local features.
* Pooling Layers: Measure half of the qubits. The measurement outcomes determine the "weighted" state of the remaining qubits, reducing the system size while preserving the most important "momentum" information.

## Step 4: Solving the Multi-Class Problem
MNIST has 10 classes (0-9). A single qubit measurement ($-1$ to $1$) isn't enough. We derive the output using One-Hot Expectation Values:
* We measure the expectation values of multiple qubits: $\langle Z_0 \rangle, \langle Z_1 \rangle, \dots, \langle Z_n \rangle$.
* We map these into a 10-dimensional vector. The class with the highest value (after a Softmax-like scaling) is our prediction.

## Step 5: The Optimization Loop (The Math of Learning)
Unlike random unitaries, we use the Parameter Shift Rule to calculate gradients. This is a principled way to find the "slope" of the quantum landscape without numerical approximation.$$\frac{\partial \langle M \rangle}{\partial \theta} = \frac{1}{2} \left( \langle M \rangle_{\theta + \pi/2} - \langle M \rangle_{\theta - \pi/2} \right)$$

# Looking at the problem
To transition from high-level intuition to a rigorous implementation of MNIST classification on quantum hardware, we must derive the mathematical constraints for each stage. We will focus on a 4-Qubit Principled Ansatz designed for a compressed $2 \times 2$ MNIST patch.

## 1. The Geometry of State Preparation (Encoding)If we compress a $28 \times 28$ image into 4 features ($x_0, x_1, x_2, x_3$), we must map this classical vector into a $2^4 = 16$-dimensional Hilbert space.
### Mathematical Derivation: ZZ-Feature Map
We don't just use a single rotation; we use a second-order expansion to capture spatial correlations. The unitary $U_{\Phi}(x)$ is defined as:$$U_{\Phi}(x) = \exp\left(i \sum_{S \subseteq \{1, \dots, n\}} \phi_S(x) \prod_{j \in S} P_j\right)$$

For MNIST, we specifically use the interaction terms ($S = \{i, j\}$) to represent adjacent pixels:$$\phi_{\{i, j\}}(x) = (\pi - x_i)(\pi - x_j)$$

This ensures that if two pixels are both "dark" or both "light," they create a unique entanglement phase that a simple linear model cannot see.

## 2. Deriving the Ansatz: The Symmetry Principle
Instead of random gates, we use a Hardware-Efficient SU(2) Ansatz. We derive this based on the principle that any single-qubit transformation can be decomposed into Euler rotations.
### The Layer Equation
A single "Principled Layer" $L(\theta)$ is defined as:$$L(\theta) = \left[ \bigotimes_{i=0}^{n-1} R_y(\theta_{i,y}) R_z(\theta_{i,z}) \right] \cdot \text{Entangler}$$
* $R_y, R_z$: These allow the state vector to move to any latitude or longitude on the Bloch sphere.
* The Entangler (CNOT Chain): This acts as the "Quantum Mixer." Mathematically, it performs a basis change that spreads the local information of one pixel across the entire 4-qubit register.

## 3. Measuring the Result (Expectation Value Math)
For MNIST, we need to map the quantum state back to a digit class. Let's say we are doing binary classification (Digit 0 vs Digit 1). We measure the Expectation Value of the Pauli-Z operator on the readout qubit:$$\langle Z \rangle = \langle \psi(\theta, x) | Z | \psi(\theta, x) \rangle$$

The math of the measurement collapses the $2^n$ complex amplitudes into a single real number:$$\langle Z \rangle = \sum_{k=0}^{2^n-1} (-1)^{f(k)} |c_k|^2$$

where $f(k)$ is 0 if the readout qubit is in state $|0\rangle$ and 1 if it is in state $|1\rangle$.
* If $\langle Z \rangle \approx 1$, the model predicts Digit 0.
* If $\langle Z \rangle \approx -1$, the model predicts Digit 1.

## 4. The Learning Rule: Parameter Shift
How does the model "learn"? We cannot use classical backpropagation because we cannot "see" inside the quantum state without collapsing it. Instead, we derive the gradient using the Parameter Shift Rule.

For any gate $G(\theta) = e^{-i\theta \hat{P}/2}$, the derivative of the expectation value is:$$\nabla_\theta \langle Z \rangle = \frac{\langle Z \rangle_{\theta + \pi/2} - \langle Z \rangle_{\theta - \pi/2}}{2}$$This is an exact analytical gradient. It tells us that to update the "weights" of our quantum MNIST model, we simply run the circuit twice: once with the parameter shifted forward and once backward.

## 5. Implementation Blueprint (AWS Braket)
Here is the mathematical assembly in code for the students to work on.

In [ ]:
from braket.circuits import Circuit, FreeParameter, Observable
import numpy as np

def mnist_principled_circuit(n_qubits, layers=1):
    qc = Circuit()

    # 1. ENCODING (Position Basis -> Momentum/Phase)
    for i in range(n_qubits):
        x = FreeParameter(f"x_{i}")
        qc.ry(i, x) # Angle Encoding

    # 2. ANSATZ (Learning Unitary)
    for l in range(layers):
        for i in range(n_qubits):
            # Euler rotations for maximum expressibility
            qc.ry(i, FreeParameter(f"theta_{l}_{i}_y"))
            qc.rz(i, FreeParameter(f"theta_{l}_{i}_z"))

        # Principled Entanglement (Nearest Neighbor)
        for i in range(n_qubits - 1):
            qc.cnot(i, i + 1)

    # 3. OBSERVABLE (Readout)
    qc.expectation(Observable.Z(), target=0)
    return qc

# Math Verification: 4 features, 1 layer = 4 input params + 8 weight params
n = 4
my_vqc = mnist_principled_circuit(n)
print(f"Total parameters to optimize: {len(my_vqc.parameters)}")

Exercise: The "Expressibility" Proof

Task: Using the code above, remove the qc.rz rotations and the qc.cnot gates.
* Observe: How does the range of possible $\langle Z \rangle$ values change?
* Math Question: Without $R_z$ and $CNOT$, your state is restricted to a 2D "slice" of the Bloch sphere. Can a model restricted to a single plane solve the non-linear "MNIST" separation problem?
* Answer: No. By removing these, you have reduced your Hilbert space dimension from $2^n$ back to $n$, losing the "Quantum Advantage."

# Building blocks for QML
We must present it as a unified physical and mathematical pipeline. We will move from the geometry of a single qubit to the collective momentum of a system, and finally to the principled design of a classifier like MNIST.

## Phase 1: The Building Blocks (Linear Algebra & Gates)
Every quantum operation is a rotation of a vector in a complex space called Hilbert Space.
* The State Vector: $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.
* The Gate (Unitary): A matrix $U$ that moves the vector without changing its length ($U^\dagger U = I$).
* The Pauli Gates:
  * X (NOT): Flips the state.
  * Z (Phase): Flips the sign of $|1\rangle$.
  * H (Hadamard): Creates superposition ($|0\rangle \to |+\rangle$).
## Phase 2: The Change of Basis (Position vs. Momentum)
This is the "Secret Sauce" of QML. We use the Quantum Fourier Transform (QFT) to shift data between two conjugate representations.
* Position Basis ($|x\rangle$): Localized data (e.g., a specific pixel value).
* Momentum Basis ($|p\rangle$): Information spread across the phase of the system.

Applying the QFT is mathematically equivalent to moving from "What is the value of this pixel?" to "What is the collective frequency/pattern of this system?"

## Phase 3: The QML Pipeline (Derived for MNIST)
When solving a problem like MNIST, we follow a principled four-step derivation.

### Step A: State Preparation (The Feature Map)
We map classical pixels $x$ into quantum phases. Using a ZZ-Feature Map, we encode not just the pixels, but the interactions (edges) between them.$$\phi_{i,j}(x) = (\pi - x_i)(\pi - x_j)$$
### Step B: The Principled Ansatz (The Learning Unitary)
Instead of random gates, we build the Ansatz using Euler Rotations and Entanglement Chains.
* Rotations ($R_y, R_z$): Provide the "weights" that move the state vector.
* CNOTs: Act as "Quantum Mixers" to correlate information across qubits.

### Step C: Measurement (The Observable)
We extract a classical prediction by measuring the Expectation Value of the Z-operator:$$\langle Z \rangle = \langle \psi(\theta, x) | Z | \psi(\theta, x) \rangle$$

### Step D: Optimization (The Parameter Shift Rule)
We update the model's weights by calculating the gradient. Mathematically, we "shift" the parameters forward and backward to see how the output changes:$$\nabla_\theta \langle Z \rangle = \frac{1}{2} (\langle Z \rangle_{\theta+\pi/2} - \langle Z \rangle_{\theta-\pi/2})$$

## Phase 4: Hands-On Laboratory (AWS Braket)
Students can now assemble this logic into a single Python script to categorize data.

In [ ]:
from braket.circuits import Circuit, FreeParameter, Observable
from braket.devices import LocalSimulator
import numpy as np

# 1. ENCODING: Position -> Momentum
def encode_data(n_qubits):
    qc = Circuit()
    for i in range(n_qubits):
        qc.ry(i, FreeParameter(f"x_{i}"))
    return qc

# 2. ANSATZ: Principled Learning
def build_ansatz(n_qubits):
    qc = Circuit()
    for i in range(n_qubits):
        qc.ry(i, FreeParameter(f"theta_{i}_y"))
        qc.rz(i, FreeParameter(f"theta_{i}_z"))
    # Entangle qubits to capture correlations
    for i in range(n_qubits - 1):
        qc.cnot(i, i + 1)
    return qc

# 3. COMPLETE PIPELINE
n = 4 # for a 2x2 MNIST patch
full_circuit = encode_data(n).add_circuit(build_ansatz(n))
full_circuit.expectation(Observable.Z(), target=0)

# Simulate
device = LocalSimulator()
# Inject random data and weights
params = {p.name: np.random.uniform(0, np.pi) for p in full_circuit.parameters}
result = device.run(full_circuit, shots=0, inputs=params).result()

print(f"Final Prediction Score: {result.values[0]:.4f}")

# What did we learn
To build a Quantum Machine Learning (QML) system—moving from a physical problem like MNIST classification to a working quantum circuit—we follow a rigorous derivation. This process ensures that the Ansatz is not a "black box" but a principled mathematical model of the data.

## Step 1: The Problem to Theory Mapping
The Problem: We want to classify images (MNIST digits).
The Theory: We treat the pixel values as Position coordinates. To find patterns, we must move these coordinates into a space where they can interfere with each other. This is the Momentum basis.

## Step 2: Deriving the State Preparation (Encoding)
We must map classical features $x$ into a quantum state $|\psi(x)\rangle$. A principled choice for image data is the ZZ-Feature Map.
* Logic: MNIST digits are defined by the relationship between pixels (edges). The ZZ-Map encodes these correlations as entanglement.
* The Math: 1. Apply $R_y(x_i)$ to encode the individual pixel value.2. Apply $CNOT \to R_z(x_i, x_j) \to CNOT$ to encode the interaction between adjacent pixels.

## Step 3: Deriving the Principled Ansatz
The Ansatz $U(\theta)$ is the "learning" part of the circuit. We derive its structure using two principles: Expressibility and Entrainment.
* The Weights (Rotation Layer): We use Euler rotations ($R_y$ and $R_z$).
  * Math: $R(\theta) = \exp(-i \theta \hat{P}/2)$. This allows the state vector to reach any point on the Bloch sphere.
* The Correlation (Entanglement Layer): We use CNOT gates.
  * Math: These gates perform a basis change that prevents the qubits from being "separable." This allows the model to learn that "Pixel A is dark only if Pixel B is also dark."

## Step 4: The Mathematical Execution (The Pipeline)
Once the circuit is designed, we must derive the output and the learning rule.
* The Prediction: We measure the Expectation Value $\langle Z \rangle$.$$\langle Z \rangle = \langle \psi(\theta, x) | Z | \psi(\theta, x) \rangle$$

This maps the $2^n$ dimensions of Hilbert space back to a single number between -1 and 1.

* The Learning (Parameter Shift): We update the weights $\theta$ using analytical gradients. To find the gradient, we run the circuit at two points: $\theta + \pi/2$ and $\theta - \pi/2$.$$\nabla_\theta = \frac{1}{2} (\text{Output}_{forward} - \text{Output}_{backward})$$

## Phase 5: Implementation (AWS Braket Workbook)
This script assembles the foundations into a single execution flow.

In [ ]:
from braket.circuits import Circuit, FreeParameter, Observable
from braket.devices import LocalSimulator
import numpy as np

# --- 1. PROBLEM DATA ---
# Imagine a 2x2 MNIST patch
mnist_pixels = [0.1, 0.8, 0.7, 0.2]

# --- 2. THEORY: STATE PREP (POSITION TO MOMENTUM) ---
def encode_mnist(n_qubits):
    qc = Circuit()
    for i in range(n_qubits):
        qc.ry(i, FreeParameter(f"x_{i}")) # Encoding pixels as angles
    return qc

# --- 3. MATH: PRINCIPLED ANSATZ ---
def build_mnist_ansatz(n_qubits):
    qc = Circuit()
    for i in range(n_qubits):
        # Euler rotations for weights
        qc.ry(i, FreeParameter(f"w_{i}_y"))
        qc.rz(i, FreeParameter(f"w_{i}_z"))
    # Entanglement chain
    for i in range(n_qubits - 1):
        qc.cnot(i, i + 1)
    return qc

# --- 4. EXECUTION ---
n = 4
full_circuit = encode_mnist(n).add_circuit(build_ansatz(n))
full_circuit.expectation(Observable.Z(), target=0)

# Create input mapping for pixels and random initial weights
inputs = {f"x_{i}": mnist_pixels[i] for i in range(n)}
weights = {p.name: np.random.uniform(0, np.pi) for p in full_circuit.parameters if "w_" in p.name}

device = LocalSimulator()
result = device.run(full_circuit, shots=0, inputs={**inputs, **weights}).result()

print(f"MNIST Classification Score: {result.values[0]:.4f}")